# 03 — Run MDMr hill-climbing on all datasets

Mirror of notebook 02, using the R `mdmr` package on the **same CSV files**.

## Requirements

- Jupyter **R kernel** (IRkernel)
- R packages: `mdmr`, `bnlearn`, `reshape2`, `dplyr`

## Pinned hyperparameters (must match Python)

| Parameter | Value |
|-----------|-------|
| `nbf` | 15 |
| discount grid | `seq(0.5, 1.0, by = 0.01)` (51 values) |
| method | `"hc"` |
| orientation | row = parent, col = child |

## R `mdm()` API note

Before the full loop, run `?mdm` in R to confirm argument names. The `mdmr`
package uses **`nbf`** for burn-in and **`CDELT`** for the discount-factor grid
(matching Python's `delta` argument and the `CDELT` alias in `mdmp`).

If your installed `mdmr` version uses a different name (e.g. `delta`), adjust
the call in the loop cell below and document the change here.

In [1]:
library(mdmr)
library(reshape2)
library(dplyr)

source("../simulation_utils.R")

W <- 0.01
V <- 100.0
T <- 200
NBF <- 15
CDELT <- seq(0.5, 1.0, by = 0.01)
N_REPLICATIONS <- 300
DAGS <- c("3var", "5var")
DATA_DIR <- "data/"

NODE_NAMES <- list(
  "3var" = c("Y1", "Y2", "Y3"),
  "5var" = c("Y1", "Y2", "Y3", "Y4", "Y5")
)

METRIC_NAMES <- c(
  "accuracy", "sensitivity", "specificity", "ppv", "npv",
  "directional_accuracy", "computation_time", "n_edges"
)

scenario_prefix <- function(dag) {
  sprintf("dag_%s_W%s_V%s_T%s", dag, W, format(V, trim = TRUE, nsmall = 1), T)
}

data_filename <- function(dag, dataset_id) {
  sprintf("%s_ind%d.csv", scenario_prefix(dag), dataset_id)
}

true_adj_filename <- function(dag) {
  sprintf("%s_true_adjacency.csv", scenario_prefix(dag))
}

load_true_adj <- function(dag) {
  path <- file.path(DATA_DIR, true_adj_filename(dag))
  as.matrix(read.csv(path, row.names = 1))
}

adjacency_to_long <- function(dag, dataset_id, adj_mat, node_names) {
  rows <- list()
  for (i in seq_along(node_names)) {
    for (j in seq_along(node_names)) {
      rows[[length(rows) + 1]] <- data.frame(
        dag = dag,
        dataset_id = dataset_id,
        node_from = node_names[i],
        node_to = node_names[j],
        edge = as.integer(adj_mat[i, j]),
        stringsAsFactors = FALSE
      )
    }
  }
  do.call(rbind, rows)
}


Anexando pacote: 'dplyr'




Os seguintes objetos são mascarados por 'package:stats':

    filter, lag




Os seguintes objetos são mascarados por 'package:base':

    intersect, setdiff, setequal, union




In [2]:
# Pre-flight: inspect mdm() formals (run once before the big loop)
if (exists("mdm", mode = "function")) {
  print(formals(mdm))
} else {
  cat("mdm() not found — install mdmr before running this notebook.\n")
}

$data_input


$method
[1] "hc"

$...




In [3]:
adjacency_rows <- list()
metric_rows <- list()

true_adj_cache <- lapply(DAGS, load_true_adj)
names(true_adj_cache) <- DAGS

for (dag in DAGS) {
  true_adj <- true_adj_cache[[dag]]
  node_names <- NODE_NAMES[[dag]]

  for (dataset_id in seq_len(N_REPLICATIONS)) {
    data_path <- file.path(DATA_DIR, data_filename(dag, dataset_id))
    data <- read.csv(data_path)

    t0 <- proc.time()
    res <- mdm(
      data_input = data,
      method = "hc",
      nbf = NBF,
      delta = CDELT
    )
    elapsed <- (proc.time() - t0)["elapsed"]

    adj_mat <- as.matrix(res$adj_mat)
    storage.mode(adj_mat) <- "integer"

    metrics <- compute_metrics(true_adj, adj_mat)
    n_edges <- sum(adj_mat)

    adjacency_rows[[length(adjacency_rows) + 1]] <-
      adjacency_to_long(dag, dataset_id, adj_mat, node_names)

    for (metric_name in METRIC_NAMES) {
      value <- switch(
        metric_name,
        computation_time = elapsed,
        n_edges = n_edges,
        metrics[[metric_name]]
      )
      metric_rows[[length(metric_rows) + 1]] <- data.frame(
        dag = dag,
        dataset_id = dataset_id,
        metric = metric_name,
        value = as.numeric(value),
        stringsAsFactors = FALSE
      )
    }

    if (dataset_id %% 50 == 0) {
      cat(sprintf("mdmr %s: completed %d / %d\n", dag, dataset_id, N_REPLICATIONS))
    }
  }
}

r_adjacency <- do.call(rbind, adjacency_rows)
r_metrics <- do.call(rbind, metric_rows)

write.csv(r_adjacency, file.path(DATA_DIR, "r_adjacency.csv"), row.names = FALSE)
write.csv(r_metrics, file.path(DATA_DIR, "r_metrics.csv"), row.names = FALSE)

cat(sprintf("Wrote %d adjacency rows\n", nrow(r_adjacency)))
cat(sprintf("Wrote %d metric rows\n", nrow(r_metrics)))

Running bnlearn::hc() with MDM custom score...



Running bnlearn::hc() with MDM custom score...



Running bnlearn::hc() with MDM custom score...



Running bnlearn::hc() with MDM custom score...



Running bnlearn::hc() with MDM custom score...



...

Running bnlearn::hc() with MDM custom score...



Running bnlearn::hc() with MDM custom score...



mdmr 5var: completed 300 / 300


Wrote 10200 adjacency rows


Wrote 4800 metric rows
